# 05 — Avaliação única no teste (seções 17–22 e 25 do v5)

**Lê:** `selecao_modelo.json`, `painel` (treino + **teste via trava**), `panel_base` até `TRAIN_TARGET_END`, `coorte`, `perfis_treino`, `resumo_validacao`, `order_seller_revenue`.  
**Grava:** `previsoes_teste`, métricas, IC bootstrap, Diebold-Mariano, perfis, sensibilidade, tabela final.

Este é o **único** acesso confirmatório ao teste. A escolha não muda depois daqui.

In [ ]:
# === SETUP (idêntico em todos os notebooks; só muda NOTEBOOK) ===
NOTEBOOK = "05_avaliacao_teste"
RAIZ = "/content/drive/MyDrive/tcc_fluxo"      # Colab. Localmente: caminho da pasta do projeto (ou deixe None)

import os, sys
try:
    from google.colab import drive
    drive.mount("/content/drive")
    !pip install -q -r {RAIZ}/requirements.txt
except ImportError:                              # fora do Colab
    RAIZ = os.environ.get("TCC_RAIZ") or None
    if RAIZ is None:
        from pathlib import Path
        RAIZ = str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").is_file()))
sys.path.insert(0, f"{RAIZ}/src")
%load_ext autoreload
%autoreload 2

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tcc_fluxo import iniciar
C, A = iniciar(RAIZ, notebook=NOTEBOOK)

In [ ]:
from tcc_fluxo import avaliacao as AV, inferencia as INF, metricas as M, graficos, Especificacao

sel = A.carregar_json("selecao_modelo")
CHOSEN = sel["modelo_escolhido"]
spec_json = A.carregar_json("painel_spec")
spec = Especificacao.de_json(spec_json, C)
TEST_END = pd.Timestamp(A.carregar_json("corte")["TEST_END"])
train = A.carregar_treino()
test = A.carregar_teste(finalidade="confirmatoria")
hist = A.carregar_historico_treino()
resumo_val = A.carregar_df("resumo_validacao")

BOOST = sel["boost_models"]
RIDGE_NAME, RIDGE_ELIGIBLE = sel["ridge_alpha_documentado"], sel["ridge_elegivel"]
FINAL = ["Naive", "MediaMovel4", RIDGE_NAME, *BOOST]
ELIGIBLE = ["Naive", "MediaMovel4", *BOOST] + ([RIDGE_NAME] if RIDGE_ELIGIBLE else [])
print("Modelo escolhido na validação:", CHOSEN)

## 17. Reestimação com todo o treino e previsão do teste

In [ ]:
test, comparison, pred_checks = AV.prever_teste(train, test, FINAL, ELIGIBLE, spec, spec.exog)
display(pred_checks)
A.salvar_df(comparison, "previsoes_teste")
A.tabela(comparison, "previsoes_teste.csv")

det = AV.auditoria_determinismo(train, test, BOOST, spec, C.DETERMINISM_TOL)
A.tabela(det, "auditoria_determinismo.csv"); display(det)
assert det.empty or det["deterministico"].all(), "previsões não reproduzíveis no mesmo ambiente"

In [ ]:
scale_final = M.seller_scales(hist, C.TRAIN_TARGET_END)
agg_scale_final = M.aggregate_scale(hist, C.TRAIN_TARGET_END)
R = AV.metricas_teste(comparison, FINAL, ELIGIBLE, CHOSEN, scale_final, agg_scale_final)
A.tabela(R["vendedor_semana"], "metricas_teste_vendedor_semana.csv")
A.tabela(R["per_seller"], "metricas_por_vendedor.csv")
A.tabela(R["seller_summary"], "resumo_metricas_por_vendedor.csv")
A.tabela(R["agregado"], "metricas_teste_agregado.csv")
print("TESTE — nível vendedor-semana"); display(R["vendedor_semana"])
print("TESTE — total semanal da coorte"); display(R["agregado"])

### Viés do total semanal (acréscimo da v6)
Uma perda MAE estima a **mediana** condicional. Em séries com muitas semanas zeradas, a mediana fica abaixo da média e a soma das previsões dos vendedores subestima o total. Esta tabela mostra o tamanho desse efeito no total da coorte.

In [ ]:
vies = AV.vies_agregado(R["aggregate"], ELIGIBLE)
A.tabela(vies, "vies_total_semanal.csv"); display(vies)

## 18–19. Intervalos bootstrap e Diebold-Mariano (HLN)

In [ ]:
ci = INF.intervalos_bootstrap(R["per_seller"], R["aggregate"], FINAL, C.N_BOOT, C.RANDOM_SEED)
A.tabela(ci, "intervalos_confianca.csv")
display(ci.loc[ci["metrica"].isin(["skill vs Naive (agrupado)", "MASE médio por vendedor",
                                   "proporção de vendedores com ganho", "skill vs Naive"])])

dm = INF.tabela_dm(comparison, R["aggregate"], FINAL, CHOSEN, C.N_BOOT, C.RANDOM_SEED, C.MIN_WEEKS_DM)
for nivel in dm["nivel"].unique():                 # acréscimo v6: Holm dentro de cada nível
    msk = dm["nivel"].eq(nivel)
    dm.loc[msk, "p_holm"] = INF.holm(dm.loc[msk, "p_valor_hln"].to_numpy(float))
A.tabela(dm, "teste_diebold_mariano.csv"); display(dm)

## 20. Desempenho por perfil de vendedor

In [ ]:
profile = A.carregar_df("perfis_treino").set_index("seller_id")
prof = AV.metricas_por_perfil(R["per_seller"], profile, CHOSEN, ELIGIBLE)
A.tabela(prof, "metricas_por_perfil.csv")
display(prof.loc[prof["modelo"].isin(ELIGIBLE)])

## 21. Gráficos

In [ ]:
fig = graficos.agregado_teste(R["aggregate"], CHOSEN); A.figura(fig, "agregado_teste.png"); plt.show()
fig = graficos.comparacao_modelos(resumo_val, ci, FINAL, ELIGIBLE, CHOSEN, RIDGE_NAME, RIDGE_ELIGIBLE)
A.figura(fig, "comparacao_modelos.png"); plt.show()

## 22. Sensibilidade (não altera a escolha)

In [ ]:
osr = A.carregar_df("order_seller_revenue")
cohort_idx = A.carregar_df("coorte")["seller_id"]
sens, share_nd, n_sem = AV.sensibilidade(comparison, ELIGIBLE, TEST_END, scale_final, agg_scale_final, osr, cohort_idx)
print(f"Teste SEM a semana-alvo {TEST_END.date()} ({n_sem} semanas):"); display(sens)
print(f"Participação de pedidos não 'delivered' no alvo do teste (coorte): {share_nd:.2%}")
A.tabela(sens, "sensibilidade_sem_ultima_semana.csv")

## 25. Tabela final

In [ ]:
final_table = AV.tabela_final(FINAL, ELIGIBLE, CHOSEN, resumo_val, ci, R["seller_summary"], R["agregado"])
A.tabela(final_table, "tabela_final_resultados.csv"); display(final_table)
A.salvar_json({"FINAL_MODELS": FINAL, "ELIGIBLE_FINAL": ELIGIBLE, "CHOSEN_MODEL": CHOSEN,
               "scale_final": scale_final.dropna().to_dict(), "agg_scale_final": agg_scale_final}, "avaliacao_teste")

In [ ]:
A.finalizar()